# Tutorial 7: Error Handling and Resilience Patterns

## What You Will Learn

- How Scalable propagates errors from distributed workers
- Gather results with error tolerance (partial success)
- Implement retry logic with exponential backoff
- Build fault-tolerant pipeline patterns
- Analyze failure telemetry

## Prerequisites

- Completed Tutorials 1 and 6
- `pip install scalable`

In [ ]:
import os
import tempfile
import time
import json

project_dir = tempfile.mkdtemp(prefix="scalable-errors-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

# Write manifest
manifest = """\
version: 1
project:
  name: error-handling-demo
targets:
  local:
    provider: local
    max_workers: 2
    threads_per_worker: 1
    processes: false
    containers: none
components:
  worker:
    cpus: 1
    memory: 1G
tasks:
  compute:
    component: worker
"""

with open("scalable.yaml", "w") as f:
    f.write(manifest)
print("Manifest written.")

## Step 1: Understanding Error Propagation

When a function raises an exception on a Dask worker, the error is captured, serialized, and re-raised on the client.

In [ ]:
from scalable import ScalableSession


def flaky_task(scenario_id: int) -> dict:
    """A function that fails for certain inputs."""
    if scenario_id % 5 == 0:
        raise RuntimeError(f"OOM: scenario {scenario_id} exceeded memory limit")
    return {"scenario": scenario_id, "result": scenario_id * 42}


session = ScalableSession.from_yaml("./scalable.yaml", target="local")
client = session.start()

# Submit a task that will fail
future = client.submit(flaky_task, 0, tag="worker")

try:
    result = future.result()
except RuntimeError as e:
    print(f"Caught error: {type(e).__name__}: {e}")
    print("The error was raised on the worker and propagated back to the client.")

## Step 2: Gathering with Error Tolerance

Instead of failing on the first error, collect results and errors separately.

In [ ]:
from distributed import as_completed

# Submit a batch with some failures expected
futures = [client.submit(flaky_task, i, tag="worker") for i in range(15)]

succeeded = []
failed = []

for future in as_completed(futures):
    try:
        result = future.result()
        succeeded.append(result)
    except Exception as e:
        failed.append({
            "error": str(e),
            "type": type(e).__name__,
        })

print(f"Succeeded: {len(succeeded)}")
print(f"Failed: {len(failed)}")
print(f"\nFailure details:")
for f in failed:
    print(f"  [{f['type']}] {f['error']}")

## Step 3: Retry with Exponential Backoff

For transient failures, retries with backoff often succeed.

In [ ]:
import random


def sometimes_fails(scenario_id: int) -> dict:
    """Transient failure — succeeds on retry with 70% probability."""
    if random.random() < 0.3:
        raise ConnectionError(f"Timeout fetching data for scenario {scenario_id}")
    return {"scenario": scenario_id, "result": scenario_id * 42}


def submit_with_retry(client, func, *args, tag, max_retries=3, backoff=1.0):
    """Submit with exponential backoff retry."""
    last_error = None
    
    for attempt in range(max_retries + 1):
        future = client.submit(func, *args, tag=tag)
        try:
            return future.result(timeout=10)
        except Exception as e:
            last_error = e
            if attempt < max_retries:
                wait = backoff * (2 ** attempt)
                print(f"  Attempt {attempt+1} failed: {e}. Retrying in {wait:.1f}s...")
                time.sleep(wait)
    
    raise last_error


# Run with retries
random.seed(42)
print("Submitting with retry logic:")

results = []
permanent_failures = []

for i in range(5):
    try:
        r = submit_with_retry(client, sometimes_fails, i, tag="worker", max_retries=3, backoff=0.1)
        results.append(r)
        print(f"  Scenario {i}: success")
    except Exception as e:
        permanent_failures.append({"scenario": i, "error": str(e)})
        print(f"  Scenario {i}: PERMANENT FAILURE after retries")

print(f"\nCompleted: {len(results)}, Permanent failures: {len(permanent_failures)}")

## Step 4: When to Retry vs. Fail Fast

| Failure Type | Strategy | Rationale |
|-------------|----------|----------|
| Network timeout | Retry (3x, exponential) | Transient; usually resolves |
| OOM (out of memory) | Fail fast | Persistent; same inputs will fail again |
| Worker preemption | Retry (unlimited) | External; will succeed when rescheduled |
| Input validation | Fail fast | Bug in data; retrying won't help |
| Import error | Fail fast | Container/environment issue |

In [ ]:
def classify_error(error: Exception) -> str:
    """Classify an error as retryable or permanent."""
    retryable_types = (ConnectionError, TimeoutError, OSError)
    retryable_patterns = ["timeout", "connection", "temporary", "retry"]
    
    if isinstance(error, retryable_types):
        return "retryable"
    
    msg = str(error).lower()
    if any(p in msg for p in retryable_patterns):
        return "retryable"
    
    return "permanent"


# Test classification
test_errors = [
    ConnectionError("Connection timeout"),
    RuntimeError("OOM: exceeded memory limit"),
    ValueError("Invalid input parameter"),
    OSError("Temporary failure in name resolution"),
]

for err in test_errors:
    classification = classify_error(err)
    print(f"  {type(err).__name__}: '{err}' → {classification}")

## Step 5: Fault-Tolerant Pipeline Pattern

Combine partial success, retry, and caching for production resilience.

In [ ]:
from scalable import cacheable

os.environ["SCALABLE_CACHE_DIR"] = os.path.join(project_dir, "cache")


@cacheable(return_type=dict, scenario_id=int)
def cached_simulation(scenario_id: int) -> dict:
    """Cached — won't re-run on retry if previously succeeded."""
    time.sleep(0.2)
    if scenario_id == 7:
        raise RuntimeError("Scenario 7 always fails")
    return {"scenario": scenario_id, "result": scenario_id * 42}


def fault_tolerant_pipeline(n_scenarios=10, max_retries=2):
    """Run a pipeline that tolerates partial failures."""
    session = ScalableSession.from_yaml("./scalable.yaml", target="local")
    
    try:
        cl = session.start()
        
        succeeded = {}
        failed = {}
        retry_queue = [(s, 0) for s in range(n_scenarios)]
        
        while retry_queue:
            batch = retry_queue[:5]
            retry_queue = retry_queue[5:]
            
            futures = {
                cl.submit(cached_simulation, s, tag="worker"): (s, attempt)
                for s, attempt in batch
            }
            
            for future in as_completed(futures):
                scenario_id, attempt = futures[future]
                try:
                    result = future.result()
                    succeeded[scenario_id] = result
                except Exception as e:
                    if attempt < max_retries and classify_error(e) == "retryable":
                        retry_queue.append((scenario_id, attempt + 1))
                    else:
                        failed[scenario_id] = str(e)
        
        return succeeded, failed
    finally:
        session.close()


succeeded, failed = fault_tolerant_pipeline()
print(f"Pipeline complete:")
print(f"  Succeeded: {len(succeeded)}")
print(f"  Failed: {len(failed)}")
if failed:
    print(f"  Permanent failures:")
    for s, err in sorted(failed.items()):
        print(f"    Scenario {s}: {err}")

## Step 6: Graceful Session Shutdown

Always use `try/finally` to ensure telemetry is finalized.

In [ ]:
def safe_workflow():
    """Pattern: always close the session."""
    session = ScalableSession.from_yaml("./scalable.yaml", target="local")
    
    try:
        client = session.start()
        
        # Do work...
        futures = [client.submit(cached_simulation, i, tag="worker") for i in range(5)]
        
        results = []
        for future in as_completed(futures):
            try:
                results.append(future.result())
            except Exception as e:
                print(f"  Task failed: {e}")
        
        return results
    
    except Exception as e:
        print(f"Fatal error: {e}")
        return []
    
    finally:
        # ALWAYS close — finalizes telemetry
        session.close()
        print("  Session closed (telemetry finalized)")


results = safe_workflow()
print(f"Got {len(results)} results")

## Step 7: Analyzing Failure Telemetry

In [ ]:
from pathlib import Path
from collections import Counter
from scalable.telemetry.collectors import iter_run_dirs, read_jsonl

runs_dir = Path(".scalable/runs")

if runs_dir.exists():
    all_failures = []
    
    for run_dir in iter_run_dirs(runs_dir):
        failures_file = run_dir / "failures.jsonl"
        if failures_file.exists():
            all_failures.extend(read_jsonl(failures_file))
    
    if all_failures:
        print(f"Total failures across all runs: {len(all_failures)}")
        
        by_class = Counter(f.get("failure_class", "Unknown") for f in all_failures)
        print(f"\nBy type:")
        for cls, count in by_class.most_common():
            print(f"  {cls}: {count}")
    else:
        print("No failure events recorded in telemetry.")
else:
    print("No telemetry data available.")

## Step 8: Timeout Management

In [ ]:
from concurrent.futures import TimeoutError as FuturesTimeout

session = ScalableSession.from_yaml("./scalable.yaml", target="local")
client = session.start()


def slow_task(n: int) -> dict:
    """A task that takes too long."""
    time.sleep(5)  # 5 seconds
    return {"n": n}


future = client.submit(slow_task, 1, tag="worker")

try:
    # 2-second timeout
    result = future.result(timeout=2)
except Exception as e:
    print(f"Timeout handling: {type(e).__name__}")
    print(f"  Task exceeded timeout — cancelling")
    future.cancel()

session.close()
print("Session closed.")

## Summary

1. Errors propagate from workers to client via serialization
2. Use `as_completed` for partial-success gathering
3. Classify errors as retryable vs permanent
4. Exponential backoff prevents thundering herd on retries
5. `@cacheable` makes retries free for previously-succeeded tasks
6. Always use `try/finally` with `session.close()`
7. Telemetry records all failures for post-hoc analysis

## Next Steps

- **Tutorial 8**: Handle Kubernetes pod evictions
- **Tutorial 4**: Cache to make retries instantaneous
- **Tutorial 9**: ML predictions to prevent resource-related failures

In [ ]:
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
if "SCALABLE_CACHE_DIR" in os.environ:
    del os.environ["SCALABLE_CACHE_DIR"]
print("Cleaned up.")